## Preparation

In [22]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import GroupShuffleSplit, train_test_split
import pandas as pd
import pickle
import sys
import math
import random
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform
import seaborn as sns
from sklearn.metrics import silhouette_score

In [2]:
!{sys.executable} -m pip install -q "pyarrow>=13.0.0"

df = pd.read_pickle(
    r"C:\Users\Juli\Documents\Master\Projekt Genomforschung\Datasets\harmonized_data.pkl"
)

## Training data

In [3]:
# training data group split in train val
target = 'LN_IC50'
pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe', 'PosIonizable', 'NegIonizable', 'ZnBinder']
mfp_cols = [col for col in df.columns if col.startswith('Bit_')]
X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
X_chem = df[pharmacophores + mfp_cols]
y = df[target].values.astype('float32')

# split data into train and test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=6726)
idx_train_val, idx_test = next(gss_test.split(X_genomic, y, groups=df['DRUG_ID']))

# separate validation set from training set
groups_train_val = df['DRUG_ID'].iloc[idx_train_val]
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.11, random_state=42)
idx_train, idx_val = next(gss_val.split(X_genomic[idx_train_val], y[idx_train_val], groups=groups_train_val))

idx_train = idx_train_val[idx_train]
idx_val = idx_train_val[idx_val]

preprocessor = ColumnTransformer(
    transformers=[
        ('pharmacophore', StandardScaler(), pharmacophores),
        #('mfp', PCA(n_components=50, random_state=42), mfp_cols)
        ('mfp', 'passthrough', mfp_cols)
    ],
    remainder='drop'
)
gene_scaler = StandardScaler()

X_genomic_train = gene_scaler.fit_transform(X_genomic[idx_train])
X_chem_train = preprocessor.fit_transform(X_chem.iloc[idx_train])

X_genomic_val = gene_scaler.transform(X_genomic[idx_val])
X_chem_val = preprocessor.transform(X_chem.iloc[idx_val])

X_genomic_test = gene_scaler.transform(X_genomic[idx_test])
X_chem_test = preprocessor.transform(X_chem.iloc[idx_test])

# scale IC50 values
#scaler = StandardScaler()
y_train = y[idx_train]#scaler.fit_transform(y[idx_train].reshape(-1, 1)).flatten()
y_val = y[idx_val]#scaler.transform(y[idx_val].reshape(-1, 1)).flatten()
y_test = y[idx_test]#scaler.transform(y[idx_test].reshape(-1, 1)).flatten()

In [17]:
def make_loaders(batch_size=128, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        DrugResponseDataset(X_genomic_train, X_chem_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        generator=g,
    )
    val_loader = DataLoader(
        DrugResponseDataset(X_genomic_val, X_chem_val, y_val),
        batch_size=batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        DrugResponseDataset(X_genomic_test, X_chem_test, y_test),
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader, val_loader, test_loader

class DrugResponseDataset(Dataset):
    def __init__(self, X_gen, X_ch, labels):
        self.X_genomic = torch.tensor(X_gen, dtype=torch.float32)
        self.X_chem = torch.tensor(X_ch if isinstance(X_ch, np.ndarray) else X_ch.values, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_genomic[idx], self.X_chem[idx], self.y[idx]

## New: create tokens in meaningful way

In [24]:
# 1. Korrelation berechnen
gene_corr = np.corrcoef(X_genomic.T)

# 2. Distanzmatrix erstellen
dist_matrix = np.clip(1.0 - gene_corr, 0, 2)
np.fill_diagonal(dist_matrix, 0)

# 3. Ward Linkage
condensed_dist = pdist(dist_matrix)
linkage_matrix = linkage(condensed_dist, method='ward')

# 4. Automatisches Auffinden der besten Token-Anzahl (z. B. Suche von k=2 bis 50)
candidate_k = range(2, 51)
scores = []

for k in candidate_k:
    labels = fcluster(linkage_matrix, t=k, criterion='maxclust') - 1
    # Silhouette Score auf Basis der vorberechneten Distanzmatrix
    score = silhouette_score(dist_matrix, labels, metric='precomputed')
    scores.append(score)

# Optimales k auswählen
optimal_k = candidate_k[np.argmax(scores)]
print(f"Optimal gefundene Token-Anzahl: {optimal_k} (Silhouette Score: {max(scores):.4f})")

# 5. Finale Clusterzuweisung
gene_clusters = fcluster(linkage_matrix, t=optimal_k, criterion='maxclust') - 1
gene_token_indices = [np.where(gene_clusters == i)[0] for i in range(optimal_k)]

for i, indices in enumerate(gene_token_indices):
    print(f"Gene Token {i}: {len(indices)} genes assigned")

Optimal gefundene Token-Anzahl: 2 (Silhouette Score: 0.1371)
Gene Token 0: 377 genes assigned
Gene Token 1: 601 genes assigned


Another way to create gene tokens: Assign each gene to a pathway

In [30]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist

# 1. Gen-Namen aus den Spalten extrahieren
gene_feature_names = df.filter(regex=r'.* \(.*\)').columns.tolist()

# 2. Korrelationsmatrix über alle Zelllinien berechnen (wie für dein Dendrogramm)
X_genomic = df.filter(regex=r'.* \(.*\)').values.astype('float32')
gene_corr = np.corrcoef(X_genomic.T)  # (978, 978)

# 3. Distanzmatrix und Ward-Linkage berechnen
dist_matrix = np.clip(1.0 - gene_corr, 0, 2)
np.fill_diagonal(dist_matrix, 0)
condensed_dist = pdist(dist_matrix)
linkage_matrix = linkage(condensed_dist, method='ward')

# 4. Den Baum exakt in 5 Cluster (Tokens) schneiden
num_tokens = 8
gene_clusters = fcluster(linkage_matrix, t=num_tokens, criterion='maxclust') - 1

# 5. Indizes pro Token extrahieren
gene_token_indices = [np.where(gene_clusters == i)[0] for i in range(num_tokens)]

# 6. Kontrolle ausgeben
for i, indices in enumerate(gene_token_indices):
    print(f"Gen-Token {i}: {len(indices)} Gene zugeordnet")

Gen-Token 0: 185 Gene zugeordnet
Gen-Token 1: 65 Gene zugeordnet
Gen-Token 2: 127 Gene zugeordnet
Gen-Token 3: 112 Gene zugeordnet
Gen-Token 4: 85 Gene zugeordnet
Gen-Token 5: 77 Gene zugeordnet
Gen-Token 6: 150 Gene zugeordnet
Gen-Token 7: 177 Gene zugeordnet


In [25]:
import numpy as np
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import pdist

# 1. Select chemical columns
chem_cols = pharmacophores + mfp_cols

# 2. Extract unique drug feature matrix
unique_drugs_df = df.drop_duplicates(subset=['DRUG_ID'])[chem_cols]
chem_values = unique_drugs_df.values.astype(np.float64)  # shape: (n_drugs, 1032)

# 3. Compute Pearson correlation with variance check
# Identify constant columns (zero variance)
std_devs = np.std(chem_values, axis=0)
valid_mask = std_devs > 1e-8

# Compute correlation matrix
chem_corr = np.zeros((len(chem_cols), len(chem_cols)), dtype=np.float64)

# Correlate non-constant features
chem_corr_valid = np.corrcoef(chem_values[:, valid_mask].T)

# Place valid correlations into full matrix
valid_indices = np.where(valid_mask)[0]
for i_idx, i in enumerate(valid_indices):
    for j_idx, j in enumerate(valid_indices):
        chem_corr[i, j] = chem_corr_valid[i_idx, j_idx]

# 4. Convert correlation to distance: dist = 1 - r
dist_matrix_chem = 1.0 - chem_corr

# Handle any remaining NaN/Inf values by setting distance to 1.0 (uncorrelated)
dist_matrix_chem = np.nan_to_num(dist_matrix_chem, nan=1.0, posinf=1.0, neginf=1.0)
dist_matrix_chem = np.clip(dist_matrix_chem, 0.0, 2.0)
np.fill_diagonal(dist_matrix_chem, 0.0)

# 5. Hierarchical clustering on condensed distance matrix
condensed_dist = pdist(dist_matrix_chem)
linkage_chem = linkage(condensed_dist, method='ward')

# 6. Extract cluster labels for 5 tokens
num_drug_tokens = 8
chem_cluster_labels = fcluster(linkage_chem, t=num_drug_tokens, criterion='maxclust') - 1

# 7. Form index arrays for each token
chem_token_indices = [
    np.where(chem_cluster_labels == i)[0] for i in range(num_drug_tokens)
]

for i, idx in enumerate(chem_token_indices):
    print(f"Drug Token {i}: {len(idx)} features assigned")

Drug Token 0: 49 features assigned
Drug Token 1: 71 features assigned
Drug Token 2: 64 features assigned
Drug Token 3: 138 features assigned
Drug Token 4: 109 features assigned
Drug Token 5: 512 features assigned
Drug Token 6: 24 features assigned
Drug Token 7: 65 features assigned


### Molecular encoder

In [18]:
import torch
import torch.nn as nn

class ModularTokenEncoder(nn.Module):
    """Encodes disjoint subsets of feature columns into separate functional tokens."""
    def __init__(self, feature_subsets_indices, hidden_dim=64, dropout=0.3):
        super().__init__()
        # Register token index buffers
        self.subset_indices = [
            torch.tensor(idx, dtype=torch.long) for idx in feature_subsets_indices
        ]
        
        # Dedicated projection per functional feature partition
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(len(idx), hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
            for idx in feature_subsets_indices
        ])

    def forward(self, x):
        # x shape: (Batch_Size, total_features)
        token_list = []
        for idx, proj in zip(self.subset_indices, self.projections):
            sub_x = x[:, idx.to(x.device)]
            token = proj(sub_x) # (B, hidden_dim)
            token_list.append(token)
        return torch.stack(token_list, dim=1) # (B, num_tokens, hidden_dim)


class GroundedCrossAttentionModel(nn.Module):
    def __init__(
        self,
        gene_token_indices,
        chem_token_indices,
        hidden_dim=64,
        num_heads=4,
        dropout=0.3
    ):
        super().__init__()
        self.num_gene_tokens = len(gene_token_indices)
        self.num_drug_tokens = len(chem_token_indices)
        self.hidden_dim = hidden_dim

        self.input_dropout_gene = nn.Dropout(0.2)
        self.input_dropout_chem = nn.Dropout(0.3)

        # 1. Modular Token Encoders (Option B)
        self.gene_encoder = ModularTokenEncoder(gene_token_indices, hidden_dim, dropout)
        self.drug_encoder = ModularTokenEncoder(chem_token_indices, hidden_dim, dropout)

        # 2. Positional / Identity Embeddings per token
        self.gene_pos = nn.Parameter(torch.randn(1, self.num_gene_tokens, hidden_dim) * 0.02)
        self.drug_pos = nn.Parameter(torch.randn(1, self.num_drug_tokens, hidden_dim) * 0.02)

        # 3. Cross-Attention Modules
        self.cross_attn_gene_to_drug = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.cross_attn_drug_to_gene = nn.MultiheadAttention(
            embed_dim=hidden_dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )

        self.norm_gene = nn.LayerNorm(hidden_dim)
        self.norm_drug = nn.LayerNorm(hidden_dim)

        # 4. Regressor Head on Pooled Multimodal Context
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_gene, x_drug):
        batch_size = x_gene.size(0)
        x_gene = self.input_dropout_gene(x_gene)
        x_drug = self.input_dropout_chem(x_drug)

        # (B, num_tokens, hidden_dim)
        gene_tokens = self.gene_encoder(x_gene) + self.gene_pos
        drug_tokens = self.drug_encoder(x_drug) + self.drug_pos

        # Cross-Attention
        gene_attn_out, gene_attn_weights = self.cross_attn_gene_to_drug(
            query=gene_tokens, key=drug_tokens, value=drug_tokens
        )
        drug_attn_out, drug_attn_weights = self.cross_attn_drug_to_gene(
            query=drug_tokens, key=gene_tokens, value=gene_tokens
        )

        # Store weights for post-hoc interpretability
        self.last_attn_weights = {
            "gene_to_drug": gene_attn_weights.detach().cpu().numpy(),
            "drug_to_gene": drug_attn_weights.detach().cpu().numpy()
        }

        # Residual Context
        gene_context = self.norm_gene(gene_tokens + gene_attn_out)
        drug_context = self.norm_drug(drug_tokens + drug_attn_out)

        # Pool across token dimensions to prevent shortcut collapse
        gene_pooled = gene_context.mean(dim=1) # (B, hidden_dim)
        drug_pooled = drug_context.mean(dim=1) # (B, hidden_dim)

        fused = torch.cat([gene_pooled, drug_pooled], dim=1) # (B, hidden_dim * 2)
        pred = self.regressor(fused)
        return pred.squeeze(1)

In [15]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

In [31]:
# ---------------------------------------------------------------------------
# 1. Konfiguration & Seed-Liste
# ---------------------------------------------------------------------------
target = 'LN_IC50'
pharmacophores = ['Donor', 'Acceptor', 'Aromatic', 'Hydrophobe', 'LumpedHydrophobe',
                   'PosIonizable', 'NegIonizable', 'ZnBinder']
mfp_cols = [col for col in df.columns if col.startswith('Bit_')]

X_genomic_raw = df.filter(regex=r'.* \(.*\)').values.astype('float32')
X_chem_raw = df[pharmacophores + mfp_cols]
y_raw = df[target].values.astype('float32')
drug_groups = df['DRUG_ID'].values

seeds = [42, 101, 1337, 6726, 9999]  # 5 verschiedene Seeds
hidden_dim = 64
dropout = 0.3
lr = 1e-3
weight_decay = 1e-3
epochs = 10
patience = 6
batch_size = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ---------------------------------------------------------------------------
# 2. Multi-Seed Trainingsschleife
# ---------------------------------------------------------------------------
seed_results = []

for run_idx, seed in enumerate(seeds):
    print(f"\n==========================================")
    print(f" Starte Seed {run_idx + 1}/{len(seeds)} (Seed-Wert: {seed})")
    print(f"==========================================")
    
    set_seed(seed)

    # 1. Daten-Splits nach DRUG_ID
    gss_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=seed)
    idx_train_val, idx_test = next(gss_test.split(X_genomic_raw, y_raw, groups=drug_groups))

    groups_train_val = drug_groups[idx_train_val]
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.11, random_state=seed)
    idx_tr_rel, idx_val_rel = next(
        gss_val.split(X_genomic_raw[idx_train_val], y_raw[idx_train_val], groups=groups_train_val)
    )
    idx_train = idx_train_val[idx_tr_rel]
    idx_val = idx_train_val[idx_val_rel]

    # 2. Skalierung (Fit ausschließlich auf Train)
    preprocessor = ColumnTransformer(
        transformers=[
            ('pharmacophore', StandardScaler(), pharmacophores),
            ('mfp', 'passthrough', mfp_cols)
        ],
        remainder='drop'
    )
    gene_scaler = StandardScaler()

    X_gen_train = gene_scaler.fit_transform(X_genomic_raw[idx_train])
    X_chem_train = preprocessor.fit_transform(X_chem_raw.iloc[idx_train])

    X_gen_val = gene_scaler.transform(X_genomic_raw[idx_val])
    X_chem_val = preprocessor.transform(X_chem_raw.iloc[idx_val])

    X_gen_test = gene_scaler.transform(X_genomic_raw[idx_test])
    X_chem_test = preprocessor.transform(X_chem_raw.iloc[idx_test])

    y_train, y_val, y_test = y_raw[idx_train], y_raw[idx_val], y_raw[idx_test]

    # 3. DataLoader
    g = torch.Generator()
    g.manual_seed(seed)

    train_loader = DataLoader(
        DrugResponseDataset(X_gen_train, X_chem_train, y_train),
        batch_size=batch_size, shuffle=True, drop_last=True, generator=g
    )
    val_loader = DataLoader(
        DrugResponseDataset(X_gen_val, X_chem_val, y_val),
        batch_size=batch_size, shuffle=False
    )
    test_loader = DataLoader(
        DrugResponseDataset(X_gen_test, X_chem_test, y_test),
        batch_size=batch_size, shuffle=False
    )

    # 4. Modell & Optimizer initialisieren
    model = GroundedCrossAttentionModel(
        gene_token_indices=gene_token_indices,
        chem_token_indices=chem_token_indices,
        hidden_dim=hidden_dim,
        num_heads=4,
        dropout=dropout
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=patience)

    best_val_rmse = float("inf")
    best_epoch = -1
    model_checkpoint_path = f"best_model_seed_{seed}.pt"

    # 5. Training
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for b_genes, b_chem, b_y in train_loader:
            b_genes, b_chem, b_y = b_genes.to(device), b_chem.to(device), b_y.to(device).view(-1)

            optimizer.zero_grad()
            preds = model(b_genes, b_chem).view(-1)
            loss = criterion(preds, b_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * b_genes.size(0)

        # Validation
        model.eval()
        val_preds, val_targets = [], []
        val_loss = 0.0
        with torch.no_grad():
            for b_genes, b_chem, b_y in val_loader:
                b_genes, b_chem, b_y = b_genes.to(device), b_chem.to(device), b_y.to(device).view(-1)
                preds = model(b_genes, b_chem).view(-1)
                loss = criterion(preds, b_y)
                val_loss += loss.item() * b_genes.size(0)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(b_y.cpu().numpy())

        total_val_loss = val_loss / len(val_loader.dataset)
        val_rmse = root_mean_squared_error(val_targets, val_preds)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch + 1
            torch.save(model.state_dict(), model_checkpoint_path)

        scheduler.step(total_val_loss)

    # 6. Test-Set Auswertung mit dem besten Checkpoint dieses Seeds
    model.load_state_dict(torch.load(model_checkpoint_path, weights_only=True))
    model.eval()

    test_preds, test_targets = [], []
    with torch.no_grad():
        for b_genes, b_chem, b_y in test_loader:
            b_genes, b_chem = b_genes.to(device), b_chem.to(device)
            preds = model(b_genes, b_chem).view(-1)
            test_preds.extend(preds.cpu().numpy())
            test_targets.extend(b_y.numpy())

    test_rmse = root_mean_squared_error(test_targets, test_preds)
    test_r2 = r2_score(test_targets, test_preds)

    print(f"--> Ergebnis Seed {seed}: Best Epoch: {best_epoch:02d} | Val RMSE: {best_val_rmse:.4f} | Test RMSE: {test_rmse:.4f} | Test R²: {test_r2:.4f}")

    seed_results.append({
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_rmse": best_val_rmse,
        "test_rmse": test_rmse,
        "test_r2": test_r2
    })

# ---------------------------------------------------------------------------
# 3. Statistische Zusammenfassung
# ---------------------------------------------------------------------------
df_results = pd.DataFrame(seed_results)

print("\n" + "="*50)
print("GESAMTAUSWERTUNG ÜBER ALLE SEEDS")
print("="*50)
print(df_results.to_string(index=False))

print("\n--- Aggregierte Kennzahlen (Mittelwert ± Std) ---")
print(f"Test RMSE: {df_results['test_rmse'].mean():.4f} ± {df_results['test_rmse'].std():.4f}")
print(f"Test R²:   {df_results['test_r2'].mean():.4f} ± {df_results['test_r2'].std():.4f}")


 Starte Seed 1/5 (Seed-Wert: 42)
--> Ergebnis Seed 42: Best Epoch: 04 | Val RMSE: 2.3377 | Test RMSE: 2.9093 | Test R²: -0.0467

 Starte Seed 2/5 (Seed-Wert: 101)
--> Ergebnis Seed 101: Best Epoch: 10 | Val RMSE: 2.0505 | Test RMSE: 2.9146 | Test R²: 0.1670

 Starte Seed 3/5 (Seed-Wert: 1337)
--> Ergebnis Seed 1337: Best Epoch: 09 | Val RMSE: 2.4224 | Test RMSE: 3.6517 | Test R²: -0.1713

 Starte Seed 4/5 (Seed-Wert: 6726)
--> Ergebnis Seed 6726: Best Epoch: 02 | Val RMSE: 2.4667 | Test RMSE: 3.1269 | Test R²: 0.1030

 Starte Seed 5/5 (Seed-Wert: 9999)
--> Ergebnis Seed 9999: Best Epoch: 01 | Val RMSE: 3.2937 | Test RMSE: 3.4390 | Test R²: -0.4572

GESAMTAUSWERTUNG ÜBER ALLE SEEDS
 seed  best_epoch  best_val_rmse  test_rmse   test_r2
   42           4       2.337672   2.909278 -0.046739
  101          10       2.050491   2.914603  0.167014
 1337           9       2.422379   3.651718 -0.171319
 6726           2       2.466708   3.126866  0.103006
 9999           1       3.293679   3.43

Ergebnis mit 5 Tokens: